# Demo | List files from Azure Blob Storage

This notebook demonstrates how to list folders and files in Azure Blob Storage. 
It explains how listing is possible in private , blob and container access levels.

Notebook flow:
1. Configure access
2. Validate connectivity
3. List available datasets
4. List subfolders and sample files for selected datasets

## List files in Public Container (no SAS-key needed)

In [ ]:
PUBLIC_CONTAINER_URL = "https://tgisdistkartserver.blob.core.windows.net/distdata"
MAX_PUBLIC_FILES_TO_SHOW = 10

In [ ]:
from azure.storage.blob import BlobPrefix, ContainerClient

public_client = ContainerClient.from_container_url(PUBLIC_CONTAINER_URL.rstrip("/"))


def list_public_top_level_folders(client: ContainerClient) -> list[str]:
    """List first-level folders in a public container."""
    folders: list[str] = []
    for item in client.walk_blobs(name_starts_with="", delimiter="/"):
        if isinstance(item, BlobPrefix):
            folder_name = item.name.strip("/")
            if folder_name:
                folders.append(folder_name)
    return sorted(set(folders))


def list_public_subfolders(client: ContainerClient, folder: str) -> list[str]:
    """List second-level folders under one first-level folder."""
    prefix = f"{folder}/"
    subfolders = set()

    for blob in client.list_blobs(name_starts_with=prefix):
        parts = blob.name.split("/")
        if len(parts) >= 2 and parts[1]:
            subfolders.add(parts[1])

    return sorted(subfolders)


def list_public_files_for_subfolder(
    client: ContainerClient,
    folder: str,
    subfolder: str,
    max_files: int,
) -> list[str]:
    """List up to max_files for one folder/subfolder pair."""
    prefix = f"{folder}/{subfolder}/"
    files: list[str] = []

    for blob in client.list_blobs(name_starts_with=prefix):
        if blob.name.endswith("/"):
            continue
        files.append(blob.name)
        if len(files) >= max_files:
            break

    return files


public_folders = list_public_top_level_folders(public_client)
public_summary_rows: list[dict[str, str | int]] = []

for folder in public_folders:
    print(f"\n===== {folder.upper()} =====")

    subfolders = list_public_subfolders(public_client, folder)
    if not subfolders:
        print("No subfolders found.")
        continue

    for subfolder in subfolders:
        files = list_public_files_for_subfolder(
            client=public_client,
            folder=folder,
            subfolder=subfolder,
            max_files=MAX_PUBLIC_FILES_TO_SHOW,
        )

        print(f"\n{subfolder} (showing up to {MAX_PUBLIC_FILES_TO_SHOW} files):")
        for file_name in files:
            print(f"  {file_name}")

        public_summary_rows.append(
            {
                "folder": folder,
                "subfolder": subfolder,
                "shown_files": len(files),
            }
        )


print("\nSummary")
print("folder                        subfolder                    shown_files")
print("---------------------------------------------------------------------")
for row in public_summary_rows:
    print(f"{row['folder']:<30}{row['subfolder']:<29}{row['shown_files']}")

## List files restrin Public Container (no SAS-key needed)

Set a SAS token before running the listing cells.

- Recommended: environment variable `SAS_TOKEN`

In [1]:
import os

CONTAINER_URL = "https://pgisnedlasting.blob.core.windows.net/data"
BASE_PREFIX = "nedlasting/"
DATASETS = ["vern", "villrein"]
MAX_FILES_PER_SUBFOLDER = 10

# Preferred source for secrets.
SAS_TOKEN = os.getenv("SAS_TOKEN", "")

if not SAS_TOKEN:
    print("SAS token is empty. Set SAS_TOKEN to list private blobs.")
else:
    print("SAS token loaded from environment variable.")

SAS token loaded from environment variable.


## Validate connectivity

The next cell sends a `HEAD` request to one known blob URL to confirm basic access and show status and file size metadata.

In [2]:
import requests

demo_blob_url = (
    "https://pgisnedlasting.blob.core.windows.net/data/"
    "nedlasting/villrein/GEOJSON/4326/villrein_11_rogaland_4326_GEOJSON.zip"
)

response = requests.head(demo_blob_url, allow_redirects=True, timeout=20)
print("HTTP status:", response.status_code)
print("Content-Length:", response.headers.get("Content-Length"))

HTTP status: 200
Content-Length: 312375


## Create client and helper functions

In [3]:
from azure.storage.blob import BlobPrefix, ContainerClient

def build_container_url(container_url: str, sas_token: str) -> str:
    """Build a container URL with optional SAS token."""
    if not sas_token:
        return container_url.rstrip("/")
    return f"{container_url.rstrip('/')}?{sas_token.lstrip('?')}"


def list_first_level_folders(client: ContainerClient, base_prefix: str) -> list[str]:
    """List folders directly below the base prefix."""
    folders: list[str] = []
    for item in client.walk_blobs(name_starts_with=base_prefix, delimiter="/"):
        if isinstance(item, BlobPrefix):
            folder_name = item.name.removeprefix(base_prefix).strip("/")
            if folder_name:
                folders.append(folder_name)
    return sorted(set(folders))


def list_subfolders(client: ContainerClient, dataset: str) -> list[str]:
    """List format-level subfolders for one dataset under nedlasting/<dataset>/."""
    prefix = f"{BASE_PREFIX}{dataset}/"
    subfolders = set()

    for blob in client.list_blobs(name_starts_with=prefix):
        parts = blob.name.split("/")
        if len(parts) >= 3:
            subfolders.add(parts[2])

    return sorted(subfolders)


def list_files_for_subfolder(
    client: ContainerClient,
    dataset: str,
    subfolder: str,
    max_files: int = 10,
) -> list[str]:
    """List up to max_files blob names for one dataset and subfolder."""
    prefix = f"{BASE_PREFIX}{dataset}/{subfolder}/"
    files: list[str] = []

    for blob in client.list_blobs(name_starts_with=prefix):
        if blob.name.endswith("/"):
            continue
        files.append(blob.name)
        if len(files) >= max_files:
            break

    return files


container_access_url = build_container_url(CONTAINER_URL, SAS_TOKEN)
client = ContainerClient.from_container_url(container_access_url)

print("Container client ready.")

Container client ready.


## List available datasets under nedlasting/

In [4]:
available_datasets = list_first_level_folders(client, BASE_PREFIX)

print("Available datasets under nedlasting/:")
for dataset in available_datasets:
    print(f"- {dataset}")

Available datasets under nedlasting/:
- anadrome_laksefisk
- art_funksjon
- art_nasjonal
- bioklimatiske_soner_seksjoner
- elvedelta
- forurenset_grunn
- friluftsliv_kartlagt
- friluftsliv_sikra
- friluftsliv_vern
- kulturlandskap
- kulturlandskap_utvalgte
- naturtyper_hb13
- naturtyper_hb19
- naturtyper_nin
- naturtyper_utvalgte
- naturtyper_verdsatte
- ramsar
- skogsmark_nin
- snoescooterloeyper
- svalb_fartsrestriksjoner
- svalb_ilandstigningslokaliteter
- svalb_motorferdsel
- tettbebyggelse
- vannforekomster
- vannomraader
- vannregioner
- vern
- vern_foreslatt
- vern_restriksjon
- villrein


## List subfolders and sample files for selected datasets

In [5]:
summary_rows: list[dict[str, str | int]] = []

for dataset in DATASETS:
    print(f"\n===== {dataset.upper()} =====")

    subfolders = list_subfolders(client, dataset)
    if not subfolders:
        print("No subfolders found or access denied.")
        continue

    for subfolder in subfolders:
        files = list_files_for_subfolder(
            client=client,
            dataset=dataset,
            subfolder=subfolder,
            max_files=MAX_FILES_PER_SUBFOLDER,
        )

        print(f"\n{subfolder} (showing up to {MAX_FILES_PER_SUBFOLDER} files):")
        for file_name in files:
            print(f"  {file_name}")

        summary_rows.append(
            {
                "dataset": dataset,
                "subfolder": subfolder,
                "shown_files": len(files),
            }
        )


print("\nSummary")
print("dataset     subfolder              shown_files")
print("----------------------------------------------")
for row in summary_rows:
    print(f"{row['dataset']:<11}{row['subfolder']:<23}{row['shown_files']}")


===== VERN =====

FILEGDB (showing up to 10 files):
  nedlasting/vern/FILEGDB/25832/vern_0301_oslo_25832_FILEGDB.zip
  nedlasting/vern/FILEGDB/25832/vern_03_oslo_25832_FILEGDB.zip
  nedlasting/vern/FILEGDB/25832/vern_1101_eigersund_25832_FILEGDB.zip
  nedlasting/vern/FILEGDB/25832/vern_1103_stavanger_25832_FILEGDB.zip
  nedlasting/vern/FILEGDB/25832/vern_1106_haugesund_25832_FILEGDB.zip
  nedlasting/vern/FILEGDB/25832/vern_1108_sandnes_25832_FILEGDB.zip
  nedlasting/vern/FILEGDB/25832/vern_1111_sokndal_25832_FILEGDB.zip
  nedlasting/vern/FILEGDB/25832/vern_1112_lund_25832_FILEGDB.zip
  nedlasting/vern/FILEGDB/25832/vern_1114_bjerkreim_25832_FILEGDB.zip
  nedlasting/vern/FILEGDB/25832/vern_1119_ha_25832_FILEGDB.zip

GEOJSON (showing up to 10 files):
  nedlasting/vern/GEOJSON/4326/vern_0000_norge_4326_GEOJSON.zip
  nedlasting/vern/GEOJSON/4326/vern_0301_oslo_4326_GEOJSON.zip
  nedlasting/vern/GEOJSON/4326/vern_03_oslo_4326_GEOJSON.zip
  nedlasting/vern/GEOJSON/4326/vern_1101_eigersund_4